In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


In [3]:
print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER GLOBAL - FASE 4 (SALING SILANG & AUTO-SKIP) 🚀 ")
print("================================================================================")

# ================================================================================
# TAHAP 1: MEMUAT SE LURUH BERKAS PICKLE DAN MERGE JADI SATU PINTU
# ================================================================================
all_fase_4_data = {}

# 1. Load File Cimut
try:
    with open('fase_4_cimut.pkl', 'rb') as f:
        all_fase_4_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Cimut.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_4_afrida.pkl', 'rb') as f:
        all_fase_4_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Afrida.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif (Jika ada file terpisah, opsional)
try:
    if os.path.exists('fase_4_hanif.pkl'):
        with open('fase_4_hanif.pkl', 'rb') as f:
            all_fase_4_data.update(pickle.load(f))
        print("✓ Berhasil memuat data hasil konversi Hanif.")
except:
    pass

 🚀 SETUP INSERT HANDLER GLOBAL - FASE 4 (SALING SILANG & AUTO-SKIP) 🚀 
✓ Berhasil memuat data hasil konversi Cimut.
⚠️ Gagal memuat file pkl Afrida: [Errno 2] No such file or directory: 'fase_4_afrida.pkl'
✓ Berhasil memuat data hasil konversi Hanif.


In [4]:
# ================================================================================
# TAHAP 2: ATUR URUTAN STRATEGIS SALING SILANG FASE 4 GLOBAL (CIMUT, AFRIDA, & HANIF)
# ================================================================================
# Urutan di bawah ini disusun ketat lintas personel demi keselamatan relasi Foreign Key!
tables_to_insert_ordered = [
    # --- BLOK A: DATA MITRA & KEMITRAAN INDUK (Karya Hanif) ---
    'mitra',                    # Master data perusahaan/lembaga kemitraan
    'mitra_progres',            # Log perkembangan hubungan kemitraan
    'kemitraan_verifikator',    # Data staff verifikator kerja sama mitra

    # --- BLOK B: DATA MASTER SISWA INDUK (Karya Hanif) ---
    'siswa',                    # Profil induk seluruh siswa yang aktif/terdaftar
    'siswa_keluar',             # Log histori pengunduran diri / kelulusan siswa
    'siswa_mitra',              # Relasi pemetaan siswa yang ditempatkan di mitra
    'siswa_mitra_keluar',       # Log penarikan/keluar siswa dari tempat mitra

    # --- BLOK C: PERIZINAN & KEPEGAWAIAN (Karya Cimut) ---
    'izin_karyawan',            # Formulir pengajuan izin/sakit karyawan
    'verifikasi_izin',          # Log persetujuan/catatan nota izin oleh atasan
    'absensi',                  # Log kehadiran harian staff via fingerprint
    'verifikasi_absensi',       # Log verifikasi absensi harian
    'karyawan_resign',          # Log pengunduran diri/keluar staff

    # --- BLOK D: PLOTTING JADWAL ACUAN AKADEMIK (Karya Afrida) ---
    'jadwal_hari',              # Master parameter hari operasional kelas
    'jadwal',                   # Master plotting jadwal belajar-mengajar utama
    'jadwal_detail',            # Detail jam, ruang, dan komparasi jadwal harian
    'jadwal_pengajar',          # Pemetaan instruktur/guru pengajar ke dalam jadwal
    'jadwal_siswa',             # Pemetaan siswa ke dalam rombongan belajar jadwal
    'kursus_siswa',             # Pilihan program kursus yang diikat siswa (Karya Hanif - butuh jadwal)

    # --- BLOK E: CATATAN AKADEMIK & AGGREGASI JURNAL (Karya Afrida) ---
    'catatan_kelas',            # Jurnal harian atau log aktivitas belajar mengajar di kelas
    'catatan_kelas_tag',        # Tagging label / kategori catatan kelas
    'catatan_mingguan'          # Rangkuman laporan perkembangan mingguan kelas
]

In [5]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT DENGAN RINGKASAN DI ATAS & DIAGNOSTIK ERROR DI BAWAH
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DI BELAKANG LAYAR
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {
                'status': 'not_found', 
                'rows': 0, 
                'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl'
            }
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {
                'status': 'empty', 
                'rows': 0, 
                'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)'
            }
            continue
            
        try:
            # Bersihkan kolom kosong murni agar tidak merusak placeholder query
            df_to_push = df_target.dropna(axis=1, how='all')
            
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            
            # Gunakan INSERT IGNORE untuk auto-skip duplikat primary key
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            # Konversi DataFrame ke Native List Python (Hancurkan tipe data NumPy)
            raw_numpy_list = df_to_push.to_numpy().tolist()
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN"] else x for x in row) 
                for row in raw_numpy_list
            ]
            
            # Eksekusi massal
            cursor.executemany(insert_query, clean_data_tuples)
            db_connection.commit()
            
            total_rows = len(clean_data_tuples)
            results[table_name] = {
                'status': 'success', 
                'rows': total_rows, 
                'msg': f'✓ {table_name}: Sukses diproses! Sebanyak {total_rows} baris sukses dimasukkan / di-skip aman.'
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'rows': 0, 
                'msg': f'✗ {table_name}: Gagal total saat insert - Alasan: {e}'
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    # 1. Cetak yang sukses dulu biar rapi
    print("🟢 TABEL YANG SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] == 'success':
            print(f"  {results[table_name]['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses)")

    print("\n🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):")
    failed_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] in ['failed', 'not_found', 'empty']:
            print(f"  {results[table_name]['msg']}")
            failed_exist = True
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada yang gagal.")
            
    print("================================================================================\n")


    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            # Taktik A: Jika Sukses, tampilkan preview standard 5 baris teratas
            if results[table_name]['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name].head(5)) 
                print("-" * 80)
                
            # Taktik B: Jika GAGAL, tembak dan kuliti struktur datanya secara transparan!
            elif results[table_name]['status'] == 'failed':
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Alasan MySQL Menolak: {results[table_name]['msg']}")
                print("-" * 50)
                print("Berikut 5 baris sampel data yang gagal dikirim, cek tipe datanya (apakah ada .0 atau string aneh):")
                display(tables_data[table_name].head(5))
                print(f"\nTipe data kolom internal DataFrame untuk tabel '{table_name}':")
                # Menampilkan tipe data internal pandas agar ketahuan mana float ghaib / objek aneh
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

In [6]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL (MENGGUNAKAN VERSI RINGKASAN ATAS)
# ================================================================================
results_fase_4 = insert_data_with_preview_and_skip_v2(
    db_connection=db_new, 
    cursor=cursor_new, 
    tables_data=all_fase_4_data, 
    ordered_list=tables_to_insert_ordered
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)

 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG SUKSES MASUK:
  ✓ mitra: Sukses diproses! Sebanyak 22 baris sukses dimasukkan / di-skip aman.
  ✓ mitra_progres: Sukses diproses! Sebanyak 296 baris sukses dimasukkan / di-skip aman.
  ✓ kemitraan_verifikator: Sukses diproses! Sebanyak 228 baris sukses dimasukkan / di-skip aman.
  ✓ siswa: Sukses diproses! Sebanyak 1469 baris sukses dimasukkan / di-skip aman.
  ✓ siswa_keluar: Sukses diproses! Sebanyak 556 baris sukses dimasukkan / di-skip aman.
  ✓ izin_karyawan: Sukses diproses! Sebanyak 957 baris sukses dimasukkan / di-skip aman.
  ✓ verifikasi_izin: Sukses diproses! Sebanyak 2107 baris sukses dimasukkan / di-skip aman.
  ✓ absensi: Sukses diproses! Sebanyak 13444 baris sukses dimasukkan / di-skip aman.
  ✓ verifikasi_absensi: Sukses diproses! Sebanyak 11 baris sukses dimasukkan / di-skip aman.
  ✓ karyawan_resign: Sukses diproses! Sebany

,id_mitra,nama_mitra,nama_instansi,nama_sekolah,alamat_mitra,nama_pimpinan,kontak_mitra,status_mitra,visi_misi,program_mitra,...,bidang_usaha,is_leapverse,status_kemitraan,tahun_bergabung,tipe_kerjasama,is_elsa,is_classin,is_mitra_leap,created_at,kode_mitra
0,2,Fiona Febianita Sulistyo,PT Delta Jaya Mas,PT Delta Jaya Mas,Gresik,Fiona Febianita Sulistyo (HRD & GA),+6282141660768,done,"<p><span style=""font-size: 10pt; font-family: ...",<p>Bussiness English &amp; Excel</p>\r\n<p>&nb...,...,Manufacturing,0,0,2023,Perluasan Bisnis,0,0,1,2023-09-04 07:06:34,M
1,3,Chelsea,CV.RABBANI,CV.RABBANI,"Jl. Ngagel Jaya No.37, Pucang Sewu, Kec. Guben...",Chelsea,+6282138601791,done,"<p><span style=""font-size: 10pt; font-family: ...",<p>EDITING VIDEO CAPCUT</p>,...,Reselling and Retail,0,0,2023,Perluasan Bisnis,0,0,1,2023-10-23 08:28:10,M
2,6,Geraldo P. Latumahina,Hartono Electronics,HARTONO ELECTRONIC,"Bukit Mas, Jalan, Kecamatan Dukuhpakis, Kota S...",Geraldo Pandega Latumahina,082250622740,done,<p>blm dikehtahui</p>,<p>Business English</p>,...,Reselling and Retail,0,0,2024,Layanan Training,0,0,1,2023-12-04 04:56:14,M
3,7,Anggi dewantoro,PT Neo Ekspor Indonesia (NEOXPI),PT NEOXPI,"SURABAYA (Royal park 1 tl 5 no 37, Surabaya Ba...",ANGGI DEWANTORO,+6285935231945,done,"<p><span style=""font-size: 10pt; font-family: ...",<p>Business english</p>,...,Food and Beverages,0,0,2023,Layanan Training,0,0,1,2024-08-27 07:02:29,M
4,8,Susanti,KB TK Budi Mulia,KB TK Budi Mulia,"Jl. Rungkut Asri Timur IX No.17, Rungkut Kidul...",Susanti,+6287854304300,done,"<p style=""box-sizing: border-box; border: 0px;...",<p>Intrakurikuler Bahasa Inggris</p>,...,Services,0,0,2024,Layanan Training,0,0,1,2024-08-27 07:20:16,M


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: MITRA_PROGRES]
--------------------------------------------------


,id_progres_mitra,id_mitra,catatan_progres_mitra,id_user,status_progres_mitra,kemitraan_mulai,kemitraan_berakhir,created_at
0,N00007,M00002,<p>Sudah dikirimkan proposal melalui Fiona</p>,U00014,connect,None,None,2023-09-04 07:29:30
1,N00008,M00002,<p>Draft MoU</p>,U00014,follow up,None,None,2023-09-04 07:41:55
2,N00009,M00002,<p>MoU signed</p>,U00014,done,None,None,2023-09-04 07:42:34
3,N00010,M00002,<p>Meeting kebutuhan kurikulum training instit...,U00014,transfer,None,None,2023-09-04 07:43:42
4,N00011,M00003,<p>Butuh follow up probing kebutuhan</p>,U00014,connect,None,None,2023-10-23 08:31:06


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: KEMITRAAN_VERIFIKATOR]
--------------------------------------------------


...

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: VERIFIKASI_ABSENSI]
--------------------------------------------------


,id_verifikasi_absensi,status_verifikasi_absensi,catatan_atasan,created_at
0,1,Disetujui,<p>dcvbhnjk</p>,2023-05-17 15:54:46
1,3,Disetujui,<p>sudah oke untuk absensi</p>,2023-07-18 18:21:38
2,4,Disetujui,<p>okee</p>,2024-11-30 00:00:00
3,5,Disetujui,,2024-10-01 00:00:00
4,6,Disetujui,<p>Sudah ACC</p>,2024-04-01 00:00:00


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: KARYAWAN_RESIGN]
--------------------------------------------------


,id_resign,id_karyawan,id_user,alasan_resign,dokumen_pendukung,status_persetujuan,status_pengiriman,created_at
0,3,U00003,U00003,menikah dan fokus rumah tangga,1760930076_1c9467f497cf1dfb4157.pdf,Diajukan,1.0,2023-04-05 16:18:11
1,4,U00001,U00001,Tidak ada keterangan,None,None,NaN,2023-04-05 16:18:11
2,11,U00011,U00011,ikut suami,1764844234_16b2d6c6fe2b556d77c4.pdf,Diajukan,1.0,2023-05-25 09:20:40
3,12,U00012,U00012,Tidak ada keterangan,None,None,NaN,2023-05-29 13:48:56
4,14,U00014,U00014,Tidak ada keterangan,None,None,NaN,2023-05-29 13:59:36


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [7]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 4 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_4 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )